# 09 - 3D SwinViT Pipeline (Path 2)

**Inputs needed:** Cropped liver volumes (Phase 2 outputs) and `configs/default.yaml`.
**Outputs produced:** SwinViT checkpoint + `data/deep_features.csv`; optional `attention/<PID>_attention.nii.gz` heatmaps for the validation cohort.
**Runtime:** GPU-bound; budget ~1 hour on a mid-range GPU for ~50 patients.


Complete deep-learning-only path from the design doc:

```
A1/A2 -> B (phase filter + DICOM->NIfTI)
      -> C (HU window + Z-score within liver mask)
      -> D (liver segmentation)
      -> E (liver bounding-box crop)
      -> G (cropped bbox -> 3D SwinViT input)
      -> J (self-attention over patches)
      -> K (deep visual vector)
```

Key risk mitigations from the design doc are baked in:
- **Data size limitation** -> WeightedRandomSampler + MONAI 3D augmentations + Focal Loss.
- **Pre-cancer signal dilution** -> SwinViT native self-attention over the whole liver bbox (no random patches).

End artifacts:
- `models/saved/best_swinvit_model.pth` and `models/saved/swinvit_deep_extractor.pth`
- `data/deep_features.csv`
- `results/training_history_swinvit.json`
- Optional `attention/<PID>_attention.nii.gz` for the validation cohort

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.utils.config import ensure_dirs, load_config, set_seed
from src.utils.logger import setup_logger

cfg = load_config(ROOT / "configs" / "default.yaml")
ensure_dirs(cfg)
set_seed(int(cfg.get("seed", 42)))
setup_logger("hcc", log_file=Path(cfg["paths"]["logs_dir"]) / "swinvit_pipeline.log")
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Stage 1 - Phase 2 preprocessing (B->E)

Same Phase 2 pipeline used by the radiomics path - skip if cropped volumes
already exist.

In [ ]:
from tqdm.notebook import tqdm
from src.data.dicom_loader import DICOMLoader
from src.data.liver_segmentation import LiverSegmentor
from src.data.preprocessing import preprocess_volume
from src.data.cropping import crop_patient

raw_dir = Path(cfg["paths"]["raw_dir"])
processed_dir = Path(cfg["paths"]["processed_dir"])
loader = DICOMLoader(raw_dir, cfg["preprocessing"]["target_phase"], processed_dir)
segmentor = LiverSegmentor(
    gpu=bool(cfg["preprocessing"]["liver_segmentation"].get("gpu", True))
)
patient_ids = sorted(p.name for p in raw_dir.iterdir() if p.is_dir())
for pid in tqdm(patient_ids, desc="Phase 2 preprocessing"):
    out_dir = processed_dir / pid
    if (out_dir / "before_cropped.nii.gz").exists() and (out_dir / "crop_metadata.json").exists():
        continue
    try:
        volume_path = loader.convert_to_nifti(pid)
        mask_path = segmentor.segment(volume_path)
        _, stats = preprocess_volume(volume_path, mask_path, cfg)
        _, _, _ = crop_patient(volume_path.parent, zscore_stats=stats)
    except Exception as exc:
        print(f"Skip {pid}: {exc}")


## Stage 2 - Dataset, weighted sampler, augmentations

The cropped liver bounding box is resized to `cfg.swin_vit.img_size` and fed to the SwinViT.
MONAI 3D augmentations + WeightedRandomSampler counter the small + imbalanced dataset.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from src.data.dataset import HCCDataset, get_3d_augmentation, get_weighted_sampler

transform = get_3d_augmentation(cfg)
dataset = HCCDataset(
    processed_dir=cfg["paths"]["processed_dir"],
    labels_csv=cfg["paths"]["labels_csv"],
    transform=transform,
    target_size=tuple(cfg["swin_vit"]["img_size"]),
)
indices = np.arange(len(dataset))
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=dataset.labels if len(set(dataset.labels)) > 1 else None,
    random_state=int(cfg.get("seed", 42)),
)
sub = HCCDataset.__new__(HCCDataset)
sub.samples = [dataset.samples[i] for i in train_idx]
sub.labels = dataset.labels[train_idx]
sampler = get_weighted_sampler(sub) if cfg["training"].get("weighted_sampler", True) else None
train_loader = DataLoader(
    Subset(dataset, train_idx.tolist()),
    batch_size=int(cfg["training"]["batch_size"]),
    sampler=sampler,
    shuffle=sampler is None,
)
val_loader = DataLoader(
    Subset(dataset, val_idx.tolist()),
    batch_size=int(cfg["training"]["batch_size"]),
    shuffle=False,
)
len(train_loader.dataset), len(val_loader.dataset)

## Stage 3 - Build the 3D SwinViT (G)

Architecture from `cfg.swin_vit`.

In [ ]:
from src.models.swin_vit import SwinViT3D
model = SwinViT3D(
    img_size=tuple(cfg["swin_vit"]["img_size"]),
    patch_size=tuple(cfg["swin_vit"]["patch_size"]),
    in_channels=int(cfg["swin_vit"]["in_channels"]),
    embed_dim=int(cfg["swin_vit"]["embed_dim"]),
    depths=tuple(cfg["swin_vit"]["depths"]),
    num_heads=tuple(cfg["swin_vit"]["num_heads"]),
    window_size=tuple(cfg["swin_vit"]["window_size"]),
    mlp_ratio=float(cfg["swin_vit"]["mlp_ratio"]),
    drop_path_rate=float(cfg["swin_vit"]["drop_path_rate"]),
    dropout=float(cfg["swin_vit"]["dropout"]),
    num_classes=int(cfg["model"]["num_classes"]),
    use_checkpoint=bool(cfg["swin_vit"]["use_checkpoint"]),
)
n_params = sum(p.numel() for p in model.parameters())
f"SwinViT params: {n_params/1e6:.2f}M, deep feature dim={model.feature_dim}"

## Stage 4 - Train SwinViT (J + K) with Focal Loss + Adam

Self-attention is what produces the patch weighting. Focal Loss + Adam from the design doc.

In [ ]:
from src.models.classifier import FocalLoss
from src.models.trainer import Trainer

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=float(cfg["training"]["learning_rate"]),
    weight_decay=float(cfg["training"]["weight_decay"]),
)
fl = cfg["training"]["focal_loss"]
loss_fn = FocalLoss(alpha=float(fl["alpha"]), gamma=float(fl["gamma"]), reduction="mean")
trainer = Trainer(model, optimizer, loss_fn, device, cfg)
summary = trainer.fit(
    train_loader,
    val_loader,
    checkpoint_path=Path(cfg["paths"]["model_save_dir"]) / "best_swinvit_model.pth",
    history_path=Path(cfg["paths"]["results_dir"]) / "training_history_swinvit.json",
)
torch.save({"model_state": model.state_dict(), "cfg": cfg}, Path(cfg["paths"]["model_save_dir"]) / "swinvit_deep_extractor.pth")
summary

## Stage 5 - Plot training history

In [ ]:
import json
import matplotlib.pyplot as plt

history = json.loads((Path(cfg["paths"]["results_dir"]) / "training_history_swinvit.json").read_text())
epochs = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(epochs, [h["train_loss"] for h in history], label="train")
ax[0].plot(epochs, [h["val_loss"] for h in history], label="val")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(epochs, [h["train_auc"] for h in history], label="train AUC")
ax[1].plot(epochs, [h["val_auc"] for h in history], label="val AUC")
ax[1].set_title("AUC"); ax[1].legend()
plt.tight_layout(); plt.show()

## Stage 6 - Export deep feature vectors (K)

These vectors are what the cross-attention fusion (notebook 10) consumes.

In [ ]:
import pandas as pd
model.eval(); rows = []
eval_dataset = HCCDataset(
    processed_dir=cfg["paths"]["processed_dir"],
    labels_csv=cfg["paths"]["labels_csv"],
    target_size=tuple(cfg["swin_vit"]["img_size"]),
)
with torch.no_grad():
    for idx in range(len(eval_dataset)):
        sample = eval_dataset[idx]
        image = sample["image"].unsqueeze(0).to(device)
        deep = model.extract_features(image).squeeze(0).cpu().numpy()
        row = {"patient_id": sample["patient_id"], "label": int(sample["label"].item())}
        row.update({f"deep_{i}": float(v) for i, v in enumerate(deep)})
        rows.append(row)
deep_df = pd.DataFrame(rows)
deep_df.to_csv(ROOT / "data" / "deep_features.csv", index=False)
deep_df.shape

## Stage 7 - Self-attention sanity check

Reverse-map the SwinViT attention map for one validation patient back to the
original CT (using `crop_metadata.json`) and confirm it lights up inside the liver.

In [ ]:
import nibabel as nib
from src.utils.visualization import plot_attention_heatmap

if len(val_idx):
    sample = eval_dataset[int(val_idx[0])]
    pid = sample["patient_id"]
    image = sample["image"].unsqueeze(0).to(device)
    attn = model.get_attention_maps(image).cpu().numpy()[0]
    meta = json.loads((Path(cfg["paths"]["processed_dir"]) / pid / "crop_metadata.json").read_text())
    volume = nib.load(str(Path(cfg["paths"]["processed_dir"]) / pid / "before.nii.gz"))
    out = Path(cfg["paths"]["attention_dir"]) / f"{pid}_attention.nii.gz"
    plot_attention_heatmap(volume.get_fdata(), attn, meta, out, affine=volume.affine)
    heat = nib.load(str(out)).get_fdata()
    z = int(np.unravel_index(np.argmax(heat), heat.shape)[-1])
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(volume.get_fdata()[..., z].T, cmap="gray", origin="lower"); ax[0].set_title("CT")
    ax[1].imshow(volume.get_fdata()[..., z].T, cmap="gray", origin="lower")
    ax[1].imshow(heat[..., z].T, cmap="jet", alpha=0.45, origin="lower")
    ax[1].set_title("SwinViT attention")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()